In [1]:
from __future__ import print_function, division
import sys,os
# line 4 and line 5 below are for development purposes and ca be removed
qspin_path = os.path.join(os.getcwd(),"../../")
sys.path.insert(0,qspin_path)
from quspin.operators import hamiltonian # Hamiltonians and operators
#from quspin.basis import spinless_fermion_basis_1d # Hilbert space fermion basis
from quspin.basis import spin_basis_1d # Hilbert space spin basis
import numpy as np # generic math functions
import matplotlib.pyplot as plt # plotting library
from scipy.linalg import expm, sinm, cosm
from numpy.linalg import multi_dot
from scipy.sparse import csr_matrix, csc_matrix, coo_matrix, lil_matrix
from scipy.sparse.linalg import inv, eigs
from scipy.linalg import svdvals
from scipy.stats import unitary_group
from scipy import linalg
from numpy import linalg as LA
import pylab
import time
import numba
from numba import njit,c16,i8,u2
import sys
import math

np.set_printoptions(precision=2)
np.set_printoptions(suppress=True)

def initial_stabilizer_state(Ld):
    #print(Ld)
    MRi=np.zeros((Ld,2*Ld+1),dtype='uint32')
    # initial stabilizer +Y for all j
    for k in range(Ld):
        #if k%8==0: # or k==3*Ld//8 or k==5*Ld//8 or k==7*Ld//8:
        #    MRi[k][Ld+k]=1
            #print(k)
        #else:
        MRi[k][k]=1
        MRi[k][2*Ld]=0  # factor 0: +1, 1: +i, 2: -1, 3: -i
        
    #print(MRi)
    return MRi

#################################
def measurement_op(dMR,Ld,Gcd,mp,prob_d_list):
    Meo=np.zeros(2*Ld+1,dtype='uint32')
    
    #all-to-all XX
    if mp==1:
        # select pair sites 
        site=np.arange(Ld)
        pair_site0=np.random.choice(site, size=1, replace=False)
        distance = np.random.choice(d_list,size=None,replace=True, p=prob_d_list)
        distance = distance.astype('int64')
        pair_site1=(pair_site0+distance)%Ld
        if pair_site0 > pair_site1:
            tk2=pair_site0
            tk=pair_site1
        else:
            tk=pair_site0
            tk2=pair_site1
        #site_list=np.arange(1,Ld-1)
        #ll = np.random.choice(site_list, 1, replace=True)
        #print(ll)
        Meo[tk]=1 
        Meo[tk2]=1 #XX
        #Meo[ll+1+Ld]=1 #XX
        # outcome 
        todo=[0,2]
        prob_list = [0.5,0.5]
        outcome = np.random.choice(todo,size=None,replace=True, p=prob_list)
        Meo[2*Ld] = outcome #factor +1 or -1 
    
    #all-to-all YY
    if mp==2:
        # select pair sites 
        site=np.arange(Ld)
        pair_site0=np.random.choice(site, size=1, replace=False)
        distance = np.random.choice(d_list,size=None,replace=True, p=prob_d_list)
        distance = distance.astype('int64')
        pair_site1=(pair_site0+distance)%Ld
        if pair_site0 > pair_site1:
            tk2=pair_site0
            tk=pair_site1
        else:
            tk=pair_site0
            tk2=pair_site1
        #site_list=np.arange(Ld)
        #ll = np.random.choice(site_list, 1, replace=True)
        Meo[tk]=1 
        Meo[tk2]=1
        Meo[tk+Ld]=1 
        Meo[tk2+Ld]=1 #YY
        # outcome 
        todo=[0,2]
        prob_list = [0.5,0.5]
        outcome = np.random.choice(todo,size=None,replace=True, p=prob_list)
        Meo[2*Ld] = outcome #factor +1 or -1 
        
    #all-to-all ZZ
    if mp==3:
        # select pair sites 
        site=np.arange(Ld)
        pair_site0=np.random.choice(site, size=1, replace=False)
        distance = np.random.choice(d_list,size=None,replace=True, p=prob_d_list)
        distance = distance.astype('int64')
        pair_site1=(pair_site0+distance)%Ld
        if pair_site0 > pair_site1:
            tk2=pair_site0
            tk=pair_site1
        else:
            tk=pair_site0
            tk2=pair_site1
        #site_list=np.arange(Ld)
        #ll = np.random.choice(site_list, 1, replace=True)
        Meo[tk+Ld]=1 
        Meo[tk2+Ld]=1 #ZZ
        # outcome 
        todo=[0,2]
        prob_list = [0.5,0.5]
        outcome = np.random.choice(todo,size=None,replace=True, p=prob_list)
        Meo[2*Ld] = outcome #factor +1 or -1 
    
    # Meo without outcome sign
    MeoF=np.zeros(2*Ld,dtype='uint32')
    MeoF=Meo[0:2*Ld]
    
    # dMR without foctor
    dMRF=np.zeros((Ld,2*Ld),dtype='uint32')
    dMRF=dMR[:,0:2*Ld]
    #check anti commtation
    Mgs=np.dot(dMRF,np.dot(Gcd,MeoF.T))
    Mgs=Mgs%2  ## check: reduce Z2 value 
    #print(Mgs)
    
    antic_index_list=[]
    for k2 in range(Ld):
        if Mgs[k2] != 0:
            antic_index_list=np.append(antic_index_list,k2)
    
    #print(antic_index_list)
    lenMe=len(antic_index_list)
    if lenMe !=0:
        kc=np.min(antic_index_list)
        kc=kc.astype('int64')
        dMR_prev=dMR[kc].copy()
        #replace stabilizer for kc
        dMR[kc]=Meo
        for k3 in range(lenMe-1):
            k4=(antic_index_list[k3+1]).astype('int64')
            # Z,X elements update
            temp2=(dMR_prev+dMR[k4,:])%2
            ### factor update
            dMR_s=dMR[k4,:]
            r1r2=factor_update(dMR_prev,dMR_s,Ld) # factor 0: +1, 1: +i, 2: -1, 3: -i 
            ###
            dMR[k4,:]=temp2
            dMR[k4,2*Ld]=r1r2
    return dMR

def factor_update(P1,P2,Ldd):
    r1 = P1[2*Ldd]
    r2 = P2[2*Ldd]
    
    r1r2d = r1 + r2
    for j in range(Ldd):
        z1j=P1[j+Ldd]
        x1j=P1[j]
        z2j=P2[j+Ldd]
        x2j=P2[j]
        
        if z1j==0 and x1j==0:
            gf=0
        if z1j==1 and x1j==1:
            gf=z2j-x2j
        if z1j==1 and x1j==0:
            gf=x2j*(1-2*z2j)
        if z1j==0 and x1j==1:
            gf=z2j*(2*x2j-1)
        r1r2d = r1r2d + gf
    r1r2d=r1r2d%4
    return r1r2d

# https://arxiv.org/abs/2204.08489
def topological_EE(MRFd,Ld):
    # partition #
    # A | B | D | C #
    #
    LA=Ld//4
    LB=Ld//4
    LC=Ld//4
    LDD=Ld//4
    
    MRdAB=np.zeros((Ld,2*(LA+LB)),dtype='uint32')
    #Create trancated matrix
    #AB
    for k2 in range((LA+LB)):
        MRdAB[:,k2]=MRFd[:,k2].copy()
        MRdAB[:,(LA+LB)+k2]=MRFd[:,(Ld)+k2].copy()
    rankAB=rank_mod2_v2(MRdAB)
    SAB=rankAB-(LA+LB)
    return SAB

# https://arxiv.org/abs/2204.08489
def rank_mod2_v2(MRdd):
    # MRdA construct
    i = 0
    Ic = MRdd.shape[0]
    Jc = MRdd.shape[1]
    for j in range(Jc):
        ### pivot swap ###
        a1c_list=[]
        #find first
        for kk in range(i,Ic):
            a1c=MRdd[kk,j] #check: kk runs
            if a1c !=0:
                a1c_list=np.append(a1c_list,kk)
        if len(a1c_list) !=0:
            p=np.min(a1c_list)
            p=p.astype('int64')
            temp = MRdd[i].copy()
            MRdd[i] = MRdd[p].copy()
            MRdd[p,:] = temp
            ##################
            #elimination
            for k in range(Ic):
                if MRdd[k,j] ==1:
                    if i!=k:
                        MRdd[k,:]=np.mod(MRdd[k,:]+MRdd[i,:],2)
            i=i+1
    my_rank = 0
    for i in range(Ic):
        rankc_list=[]
        for k in range(Jc):
            if MRdd[i,k] != 0:
                rankc_list=np.append(rankc_list,MRdd[i,k])
        if len(rankc_list)!=0:
            my_rank +=1
    return my_rank

#####Negativity ABC#########
def negativity_AB(MRdn,Ld):
    #print(MRdn)
    #Gaussian elimination for standard form
    # MRdA construct
    i = 0
    Ic = MRdn.shape[0]
    Jc = MRdn.shape[1]
    for j in range(Jc):
        ### pivot swap ###
        a1c_list=[]
        #find first
        for kk in range(i,Ic):
            a1c=MRdn[kk,j] #check: kk runs
            if a1c !=0:
                a1c_list=np.append(a1c_list,kk)
        if len(a1c_list) !=0:
            p=np.min(a1c_list)
            p=p.astype('int64')
            temp = MRdn[i].copy()
            MRdn[i] = MRdn[p].copy()
            MRdn[p,:] = temp
    #        ##################
            #elimination
            for k in range(Ic):
                if MRdn[k,j] ==1:
                    if i!=k:
                        MRdn[k,:]=np.mod(MRdn[k,:]+MRdn[i,:],2)
            i=i+1
    #####################################
    #ACBC
    # Create G(AB)
    #Check AC stabilizer
    LA=Ld//4
    LB=Ld//4
    LC=Ld//4
    # C = bar (A or B)
    MRAB=np.zeros((1,2*Ld),dtype='uint32')
    mAB=0
    for k in range(Ld):
        gstab=MRdn[k,:]
        #print(k,gstab)
        Xc=0
        Zc=0
        for k1 in range(LC):
            Xc=Xc+gstab[LA+k1]
            Zc=Zc+gstab[LA+Ld+k1]
            Xc=Xc+gstab[3*LA+k1]
            Zc=Zc+gstab[3*LA+Ld+k1]
        #print(k,Xc,Zc)
        if Xc ==0 and Zc ==0:
            if mAB==0:
                MRAB[0][:]=gstab
                mAB=mAB+1
            else:
                MRAB=np.vstack((MRAB,gstab))
                mAB=mAB+1
    # Restriction A
    MRA=np.zeros((mAB,2*(LA)),dtype='uint32')
    for k2 in range(LA):
        MRA[:,k2]=MRAB[:,k2].copy()
        MRA[:,LA+k2]=MRAB[:,(Ld)+k2].copy()
    #print(888,MRA)  
    #check matrix
    GcdA=np.zeros((2*LA,2*LA),dtype='uint32') # 2L * 2L matrix
    for k in range(LA):
        GcdA[k][LA+k]=1
        GcdA[LA+k][k]=1  

    # Create commutator matrix
    ComM=np.zeros((mAB,mAB),dtype='uint32')
    for i in range(mAB):
        for j in range(mAB):
        #check anti commtation
            gstabi=MRA[i,:]
            gstabj=MRA[j,:]
            acomgigj=np.dot(gstabi,np.dot(GcdA,gstabj.T))
            acomgigj=acomgigj%2  ## check: reduce Z2 value
            ComM[i,j]=acomgigj
    print(85,ComM)
    rankJ=rank_mod2_v2(ComM)
    NA=0.5*rankJ
    #print(NA)
    return NA

### Mutual I_AB ###
def mutual_AB(MRFd,Ld):
    # https://arxiv.org/abs/2204.08489
    # partition #
    # A | C| B | C #
    #
    LA=Ld//4
    LB=Ld//4 #Ld//3
    LC=Ld//4 #Ld//3

    MRdA=np.zeros((Ld,2*(LA)),dtype='uint32')
    MRdB=np.zeros((Ld,2*(LB)),dtype='uint32')
    MRdC=np.zeros((Ld,2*LC),dtype='uint32')
    MRdAB=np.zeros((Ld,2*(LA+LB)),dtype='uint32')
    MRdAC=np.zeros((Ld,2*(LA+LC)),dtype='uint32')

    #Create trancated matrix
    #A
    for k2 in range(LA):
        MRdA[:,k2]=MRFd[:,k2].copy()
        MRdA[:,(LA)+k2]=MRFd[:,(Ld)+k2].copy()
    #B    
    for k2 in range(LB):
        MRdB[:,k2]=MRFd[:,LA+LC+k2].copy()
        MRdB[:,LB+k2]=MRFd[:,(Ld)+LA+LC+k2].copy()
    #AB
    for k2 in range(LA):
        MRdAB[:,k2]=MRFd[:,k2].copy()
        MRdAB[:,LA+LB+k2]=MRFd[:,Ld+k2].copy()
    for k2 in range(LB):
        MRdAB[:,LA+k2]=MRFd[:,LA+LC+k2].copy()
        MRdAB[:,2*LA+LB+k2]=MRFd[:,Ld+LA+LC+k2].copy()
    rankA=rank_mod2_v2(MRdA)
    SA=rankA-(LA)
    rankB=rank_mod2_v2(MRdB)
    SB=rankB-(LB)
    rankAB=rank_mod2_v2(MRdAB)
    SAB=rankAB-(LA+LB)
    
    I_AB=SA+SB-SAB
    return I_AB

### Main start ###
t1 = time.time()
L=24 #syste size
# stabilizer matrix
MR=np.zeros((L,2*L+1),dtype='uint32') # L * 2L+1 matrix, 2*L+1 = constant factor 
MRF=np.zeros((L,2*L),dtype='uint32') # L * 2L matrix
# factor cut stabilizer matrix
#check matrix
Gc=np.zeros((2*L,2*L),dtype='uint32') # 2L * 2L matrix
for k in range(L):
    Gc[k][L+k]=1
    Gc[L+k][k]=1
Nd = 200
#### tree like coupling parameter ####
alpha=0
normp=0
d_list=[]
prob_d_list=[]
for d in range(1,L//2+1,1):
    normp=normp+(d**alpha)
    d_list=np.append(d_list,d)
for d in range(1,L//2+1,1):
    prob_d_list = np.append(prob_d_list,(d**alpha)/normp)
#################################

Stopo_ave=[]
SAB_list=[]
NAB_list=[]
IAB_list=[]
### time evolution ###
tmax=20
Nt=tmax*L #8*L #15*
Ntt=0 #10*L #660*L #4*Nt//5
tE_list=[]
for ids in range(Nd):
    MR=initial_stabilizer_state(L)
    MRF=MR[:,0:2*L]
    for it in range(Nt):
        if it==0:
            MRF=MR[:,0:2*L] # eliminate factor elements
        if it !=0:
            todo=[1,2,3]
            prob_list = [1/3,1/3,1/3]
            mtype= np.random.choice(todo,size=None,replace=True, p=prob_list)
            MR=measurement_op(MR,L,Gc,mtype,prob_d_list)
            #if it==Nt-1:
            #topological EE
            MRF=MR[:,0:2*L] # eliminate factor elements
            if it==Nt-1:
                NABd=negativity_AB(MRF,L)
                IABd=mutual_AB(MRF,L)
                print(ids,NABd)
                NAB_list=np.append(NAB_list,NABd)
                IAB_list=np.append(IAB_list,IABd)

Nave2=np.mean(NAB_list)
Iave2=np.mean(IAB_list)

print(Nave2,Iave2)


/tmp/ipykernel_41087/672910215.py:179: RuntimeWarning: overflow encountered in scalar subtract
  gf=z2j*(2*x2j-1)
/tmp/ipykernel_41087/672910215.py:175: RuntimeWarning: overflow encountered in scalar subtract
  gf=z2j-x2j
/tmp/ipykernel_41087/672910215.py:180: RuntimeWarning: overflow encountered in scalar add
  r1r2d = r1r2d + gf
/tmp/ipykernel_41087/672910215.py:177: RuntimeWarning: overflow encountered in scalar subtract
  gf=x2j*(1-2*z2j)


85 [[0]]
0 0.0
85 []
1 0.0
85 []
2 0.0
85 []
3 0.0
85 [[0]]
4 0.0
85 [[0 0]
 [0 0]]
5 0.0
85 [[0 1]
 [1 0]]
6 1.0
85 [[0]]
7 0.0
85 []
8 0.0
85 []
9 0.0
85 []
10 0.0
85 [[0 0]
 [0 0]]
11 0.0
85 [[0 0 0]
 [0 0 0]
 [0 0 0]]
12 0.0
85 []
13 0.0
85 [[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
14 0.0
85 [[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]
15 0.0
85 []
16 0.0
85 [[0]]
17 0.0
85 []
18 0.0
85 [[0]]
19 0.0
85 [[0]]
20 0.0
85 []
21 0.0
85 []
22 0.0
85 [[0]]
23 0.0
85 []
24 0.0
85 [[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]
25 0.0
85 [[0 0 0]
 [0 0 0]
 [0 0 0]]
26 0.0
85 [[0 1]
 [1 0]]
27 1.0
85 []
28 0.0
85 []
29 0.0
85 [[0]]
30 0.0
85 [[0 0]
 [0 0]]
31 0.0
85 []
32 0.0
85 []
33 0.0
85 []
34 0.0
85 [[0 0 0]
 [0 0 0]
 [0 0 0]]
35 0.0
85 []
36 0.0
85 [[0]]
37 0.0
85 []
38 0.0
85 []
39 0.0
85 [[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]
40 0.0
85 [[0 1]
 [1 0]]
